# Executive Summary

In this notebook, I will do a step-by-step exploration of the News API v2/everything endpoint, building from bare keyword lookup to multi-operator domain filter queries.

I then try to apply this to 12 representative companies extracted from the 96 company database.

I will also save any extracted results under `data/raw/search_cache` For future reference and to create a historical collection.

IMPORTANT DESIGN CHOICHES:
- Given Vishal's previous limitations in selecting and finding news for SMEs, I will try a different approach and create a **scoring heuristic** that rewards companies that are more likely to be covered by the news. I hereby acknowledge that this is going to bias the results, so I will also include a randomly selected sample with a low heuristic.

### 1. Setup

In [7]:
import importlib, subprocess, sys

def ensure(pkg, import_as=None):
    try: importlib.import_module(import_as or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('beautifulsoup4', 'bs4')
ensure('lxml')

In [8]:
import os, json, time, re
from pathlib import Path
from datetime import datetime, timedelta, date
from urllib.parse import urlencode, quote_plus, urlparse, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv('../.env')

NEWS_API_KEY = os.getenv('NEWS_API_KEY')
assert NEWS_API_KEY, 'NEWS_API_KEY missing from .env'

TODAY      = date.today()
MONTH_AGO  = TODAY - timedelta(days=28)   # stay inside free-tier 30-day window
EVERYTHING_URL = 'https://newsapi.org/v2/everything'

print(f'Key loaded. Window: {MONTH_AGO} → {TODAY}')

Key loaded. Window: 2026-06-07 → 2026-07-05


In [9]:
CACHE_DIR = Path('../data/raw/search_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

_request_count = 0

def cached_get(url: str, params: dict, company_number: str, label: str) -> dict:
    '''GET with disk cache. Never hits the API twice for the same (company_number, label).'''
    global _request_count
    cache_path = CACHE_DIR / f'{company_number}_newsapi_{label}.json'
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    _request_count += 1
    resp = requests.get(url, params=params,
                        headers={'X-Api-Key': NEWS_API_KEY}, timeout=10)
    data = resp.json()
    cache_path.write_text(json.dumps(data, indent=2))
    print(f'  [call #{_request_count}] {company_number}/{label} '
          f'→ status={data.get("status")} total={data.get("totalResults","?")} ')
    return data

def show_articles(data: dict, n: int = 5):
    '''Pretty-print top-n articles from a NewsAPI response dict.'''
    if data.get('status') != 'ok':
        print(f'ERROR: {data.get("code")} — {data.get("message")}')
        return
    total = data.get('totalResults', 0)
    arts  = data.get('articles', [])
    print(f'totalResults: {total}  (showing {min(n, len(arts))})')
    print('-' * 80)
    for a in arts[:n]:
        pub  = a.get('publishedAt', '')[:10]
        src  = (a.get('source') or {}).get('name', 'unknown')
        desc = (a.get('description') or '')[:120]
        print(f'[{pub}] {src}')
        print(f'  {a.get("title", "")}')
        print(f'  {desc}')
        print()

### 2. Company Selection + Heuristic

The heuristic follows this list:

1. If company.category = "public limited company", it gets 4 points because PLCs have disclosure obligations and are almost guaranteed to be covered by the press.
2. If account.category = group, it will give 3 points because the company files consolidated group accounts = is the parent of a corporate group with subsidiaries = more coverage signal
3. If num.mort.charges > 0 the company gets 2 points because registered charge = financial hitory = more business events = more press
4. Age will be a continous normalised value 0-1 as older firm have longer press history to draw from

In [10]:
df = pd.read_csv('../data/processed/nb04_stress_test_sample.csv')
df.columns = [c.strip() for c in df.columns]

df['_plc']   = (df['CompanyCategory'] == 'Public Limited Company').astype(int) * 4
df['_group'] = (df['Accounts.AccountCategory'] == 'GROUP').astype(int) * 3
df['_mort']  = (df['Mortgages.NumMortCharges'] > 0).astype(int) * 2
df['_inc']   = pd.to_datetime(df['IncorporationDate'], dayfirst=True, errors='coerce')
df['_age']   = (pd.Timestamp(str(TODAY)) - df['_inc']).dt.days / 365.25

picks_list = []
for (sector, segment), grp in df.groupby(['sector', 'segment']):
    grp = grp.copy()
    mn, mx = grp['_age'].min(), grp['_age'].max()
    grp['_age_n'] = (grp['_age'] - mn) / (mx - mn) if mx > mn else 0.0
    grp['_score'] = grp['_plc'] + grp['_group'] + grp['_mort'] + grp['_age_n']
    picks_list.append(grp.loc[grp['_score'].idxmax()])

picks = pd.DataFrame(picks_list).reset_index(drop=True)
disp = picks[['CompanyName','CompanyNumber','RegAddress.PostTown','sector','segment',
              '_age','_plc','_group','_mort','_score']]
disp.columns = ['Name','Number','Town','Sector','Tier','Age','PLC','Grp','Mort','Score']
print(disp.to_string(index=False))

                                 Name   Number           Town                           Sector   Tier        Age  PLC  Grp  Mort     Score
                           WHALAR LTD 09803195         LONDON           Fast growth & emerging  Large  10.759754    0    3     2  5.560240
RADIO COMPUTING SERVICES (UK) LIMITED 02844235      GODALMING           Fast growth & emerging Medium  32.898015    0    0     2  3.000000
  J.C. COMPUTER SERVICES (UK) LIMITED 02254984         HARROW           Fast growth & emerging  Micro  38.151951    0    0     2  3.000000
                         VEREMARK LTD 11681510         LONDON           Fast growth & emerging  Small   7.633128    0    0     2  2.135820
                       ZENTIA LIMITED 00207732    TYNE & WEAR                    Manufacturing  Large 100.908966    0    0     2  3.000000
        SHEPCOTE DISTRIBUTORS LIMITED 00949706      DRIFFIELD                    Manufacturing Medium  57.316906    0    0     2  3.000000
                PECO SERVIC

### 3. NewsAPI Architecture

News API has three types of endpoints:

1. V2 Everything, which searches all 150,000 indexed articles.
2. V2 Top Headlines, which covers breaking news for a country or a category.
3. V2 Top Headlines Sources, which gives metadata about available publishers.

#### 3.1 Level 1 - Bare query

I start with the simplest possible call, which queries the name (`q={name}`). I try with the company that scored the highest heuristic, NCC Group.

In [11]:
# NCC Group PLC: highest-confidence pick (PLC, cybersecurity, FTSE-listed)
data_ncc_l1 = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'NCC Group', 'pageSize': 10},
    company_number='04627044',
    label='L1_bare'
)
show_articles(data_ncc_l1)

totalResults: 53  (showing 5)
--------------------------------------------------------------------------------
[2026-06-11] MarketBeat
  NCC Group H1 Earnings Call Highlights
  NCC Group (LON:NCC) said it has completed a major strategic reset after selling Escode and is now operating as a focused

[2026-06-24] Infosecurity Magazine
  Iran-Linked MuddyWater Poses as Ransomware Gang to Mask Cyber Espionage
  An NCC Group report warns state-backed hackers are attempting to hide activity by posing as ransomware groups and deploy

[2026-06-11] Internet
  The Gentlemen Ransomware Claims 478 Victims, Can Spread Like a Worm
  A new analysis of The Gentlemen operation has revealed that the financially motivated threat group initially operated as

[2026-06-18] NPR
  As America turns 250, one museum makes history possible to touch
  Federal law requires most museums and other buildings to be accessible to people with disabilities. But access to what's

[2026-06-25] Internet
  ThreatsDay Bulletin:

In [12]:
# PECO SERVICES LIMITED — manufacturing micro-firm, Kirkby Stephen.
# 'PECO' is also PECO Energy, a large Philadelphia utility with heavy US press coverage.
# This is the canonical false-positive example throughout this notebook.
data_peco_l1 = cached_get(
    url=EVERYTHING_URL,
    params={'q': 'Peco', 'pageSize': 10},
    company_number='04623078',
    label='L1_bare'
)
show_articles(data_peco_l1)
# Expected: mostly PECO Energy (Philadelphia utility) articles.

totalResults: 42  (showing 5)
--------------------------------------------------------------------------------
[2026-05-30] Xatakamovil.com
  He creado una app para mi Android en quince minutos sin tener ni idea de programación. Ahora puedo hacer cualquier idea que me apetezca
  Cuántas veces habré buceado por Play Store buscando una app muy concreta solo para terminar descargando opciones llenas 

[2026-06-05] Prtimes.jp
  「子どもの時間感覚の育て方」「学び旅＆体験ガイド」2大特集／『AERA with Kids2026夏号』発売！／特別付録「変身！正方形パズル」／教科別おすすめドリル／巻頭インタビューpecoさん登場
  株式会社朝日新聞出版のプレスリリース（2026年6月5日 11時00分）「子どもの時間感覚の育て方」「学び旅＆体験ガイド」2大特集／『AERA with Kids2026夏号』発売！／特別付録「変身！正方形パズル」／教科別おすすめドリル／巻

[2026-06-02] PRNewswire
  Façonnons l'avenir ensemble ! Voilà ce qu'on dit les participants du monde entier lors de la réunion 2026 des dirigeants locaux Chine-PECO et de la semaine des villes amies de Shandong
  JINAN, Chine, 2 juin 2026 /PRNewswire/ -- Du 25 au 29 mai 2026, la 7e réunion des dirigeants locaux Chine-PECO et la sem

[2026-06-02] 